# CEOAI Practice 2 - Trace Twins Minimum Solution

Objective: serialize a valid `Submission` object:

1. Build local validation windows and pairs from `public_traces.csv`.
2. Use token overlap for Part A.
3. Use frequency-shape similarity for Part B, where token names may be scrambled.
4. Save `submission.pkl` with `score_A` and `score_B`.

In [1]:
from pathlib import Path
from collections import Counter
import itertools
import zipfile
import cloudpickle
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd()
DATA = ROOT / "data"
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)

In [2]:
zip_path = DATA / "public_traces.zip"
if zip_path.exists() and not (DATA / "public_traces.csv").exists():
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(DATA)

traces = pd.read_csv(DATA / "public_traces.csv")
traces.head()

,program_id,category,tokens
0,program_000,adware,p0_a p0_a open_url open_url dns_query p0_a p0_...
1,program_001,trojan,p1_a net_send reg_read inject_proc reg_read re...
2,program_002,ransom,p2_a p2_a delete_shadow p2_b write_note read_f...
3,program_003,worm,p3_a exec_remote p3_a dns_query copy_self scan...
4,program_004,adware,p4_a p4_a p4_a write_cache p4_b open_url dns_q...


In [3]:
def print_dir(obj):
    for i in dir(obj):
        if i.startswith("_"):
            continue
        print(i)

In [4]:
print_dir(np.ndarray)

T
all
any
argmax
argmin
argpartition
argsort
astype
base
byteswap
choose
clip
compress
conj
conjugate
copy
ctypes
cumprod
cumsum
data
device
diagonal
dot
dtype
dump
dumps
fill
flags
flat
flatten
getfield
imag
item
itemsize
mT
max
mean
min
nbytes
ndim
nonzero
partition
prod
put
ravel
real
repeat
reshape
resize
round
searchsorted
setfield
setflags
shape
size
sort
squeeze
std
strides
sum
swapaxes
take
to_device
tobytes
tofile
tolist
trace
transpose
var
view


In [5]:
unique = np.unique(traces["tokens"].to_numpy().data)
print(unique[0])
print(unique.shape)

p0_a p0_a open_url open_url dns_query p0_a p0_a write_cache dns_query p0_b p0_a p0_a p0_a dns_query read_cookie p0_a p0_a dns_query dns_query spawn_popup p0_a p0_a dns_query dns_query spawn_popup p0_a p0_a read_cookie dns_query spawn_popup p0_a p0_a open_url dns_query open_url p0_a p0_a write_cache spawn_popup read_cookie p0_a p0_a spawn_popup open_url spawn_popup p0_a p0_a write_cache spawn_popup read_cookie p0_a p0_a read_cookie read_cookie read_cookie p0_a p0_a open_url p0_b p0_b p0_a p0_a p0_b dns_query write_cache p0_a p0_a dns_query read_cookie spawn_popup p0_a p0_a write_cache open_url spawn_popup p0_a p0_a p0_b write_cache dns_query p0_a p0_a read_cookie open_url write_cache p0_a p0_a open_url dns_query open_url p0_a p0_a spawn_popup read_cookie dns_query p0_a p0_a open_url dns_query read_cookie p0_a p0_a read_cookie open_url spawn_popup p0_a p0_a dns_query spawn_popup read_cookie p0_a p0_a dns_query open_url write_cache p0_a p0_a open_url read_cookie spawn_popup p0_a p0_a spaw

In [6]:
def make_windows(tokens, size=200, stride=200):
    return [tokens[i:i + size] for i in range(0, len(tokens) - size + 1, stride)]

windows = []
program_for_window = []
category_for_window = []
for _, row in traces.iterrows():
    toks = row["tokens"].split()
    for window in make_windows(toks):
        windows.append(window)
        program_for_window.append(row["program_id"])
        category_for_window.append(row["category"])

by_program = {}
by_category = {}
for idx, program in enumerate(program_for_window):
    by_program.setdefault(program, []).append(idx)
    by_category.setdefault(category_for_window[idx], []).append(idx)

pairs = []
labels = []
for indexes in by_program.values():
    for i, j in zip(indexes, indexes[1:]):
        pairs.append((i, j))
        labels.append(1)

target_negatives = len(labels)
candidate_negatives = []
for i, j in itertools.combinations(range(len(windows)), 2):
    if program_for_window[i] == program_for_window[j]:
        continue
    if category_for_window[i] == category_for_window[j] or len(candidate_negatives) % 3 == 0:
        candidate_negatives.append((i, j))
rng = np.random.default_rng(0)
chosen = rng.choice(len(candidate_negatives), size=target_negatives, replace=False)
for idx in chosen:
    pairs.append(candidate_negatives[int(idx)])
    labels.append(0)
print({"windows": len(windows), "pairs": len(pairs), "positives": sum(labels)})

{'windows': 84, 'pairs': 112, 'positives': 56}


In [7]:
def jaccard_score(a, b):
    sa, sb = set(a), set(b)
    return len(sa & sb) / max(len(sa | sb), 1)

def frequency_shape(window, top_k=20):
    counts = sorted(Counter(window).values(), reverse=True)
    counts = counts[:top_k] + [0] * max(0, top_k - len(counts))
    arr = np.array(counts, dtype=float)
    return arr / max(arr.sum(), 1.0)

def repeat_signature(window):
    adjacent_repeats = sum(1 for a, b in zip(window, window[1:]) if a == b) / max(len(window) - 1, 1)
    counts = Counter(window)
    top_counts = sorted(counts.values(), reverse=True)[:8]
    top_counts = top_counts + [0] * (8 - len(top_counts))
    top_counts = np.array(top_counts, dtype=float) / len(window)
    return np.concatenate([[adjacent_repeats], top_counts])

def cosine(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom else 0.0

class Submission:
    def score_A(self, windows, pairs):
        return [float(jaccard_score(windows[i], windows[j])) for i, j in pairs]

    def score_B(self, windows, pairs):
        shapes = [np.concatenate([frequency_shape(w), repeat_signature(w)]) for w in windows]
        return [float(cosine(shapes[i], shapes[j])) for i, j in pairs]

sub = Submission()
scores_a = sub.score_A(windows, pairs)
scores_b = sub.score_B(windows, pairs)
print({
    "fixture_auc_A": float(roc_auc_score(labels, scores_a)),
    "fixture_auc_B": float(roc_auc_score(labels, scores_b)),
})

{'fixture_auc_A': 1.0, 'fixture_auc_B': 0.7633928571428571}


In [8]:
with open(OUT / "submission.pkl", "wb") as f:
    cloudpickle.dump(sub, f)

with open(OUT / "submission.pkl", "rb") as f:
    loaded = cloudpickle.load(f)
assert len(loaded.score_A(windows[:4], [(0, 1), (2, 3)])) == 2
assert len(loaded.score_B(windows[:4], [(0, 1), (2, 3)])) == 2
print("wrote", OUT / "submission.pkl")

wrote d:\projects\Supervised-Learning-Experiments\olympiads\competition_samples\raw\ceoai-2026-practice-rounds\round-2\trace_twins\outputs\submission.pkl
